In [ ]:

import warnings; warnings.filterwarnings("ignore")
import torch
import observer

torch.set_grad_enabled(False)
torch.set_default_dtype(torch.double)

ds_train_name = "ks2d_large_data" # or "ks2d_small_data"
ds_test_name = "ks2d"
n_pred = 100
n_obs = 5

C_train = torch.load(f"data/{ds_train_name}_train.pt", weights_only=True)
C_test = torch.load(f"data/{ds_test_name}_test.pt", weights_only=True)

torch.manual_seed(0) # <- makes sure that the same random weights are generated for full and convolutional EDMD 
obs = observer.Observer(n_obs=n_obs)
rec = observer.Reconstructor(obs(C_train), C_train)

loss: 5.894e-01: 100%|██████████| 100/100 [00:00<00:00, 335.16it/s]


In [74]:
class EDMD(torch.nn.Module):
    def __init__(self, obsC):
        super().__init__()
        X, Y = obsC[:-1], obsC[1:]
        X, Y = X.flatten(1), Y.flatten(1)
        self.K = torch.linalg.lstsq(X, Y, driver="gelsd", rcond=1e-5).solution.T

    def forward(self, x):
        squeezed = len(x.shape) == 3
        if squeezed: x = x.unsqueeze(0)
        shape = x.shape
        x = x.flatten(1)
        x = x @ self.K.T
        x = x.view(*shape)
        if squeezed: x = x.squeeze(0)
        return x
    
    def eigenpairs_left(self):
        return torch.linalg.eig( self.K.T )

K = EDMD(obs(C_train))

In [75]:
X_test = obs(C_test)

C_pred = torch.zeros(n_pred, *C_test.shape[1:])
X_pred = torch.zeros(n_pred, *X_test.shape[1:])
X_pred[0] = X_test[0]
C_pred[0] = rec(X_pred[0])
for j in range(n_pred-1):
    X_pred[j+1] = K( X_pred[j] )
    C_pred[j+1] = rec( X_pred[j+1] )

torch.save(C_pred, f"generated data/{ds_train_name}_full_edmd_trajectory_pred.pt")
torch.save(C_test[:n_pred], f"generated data/{ds_train_name}_full_edmd_trajectory_test.pt")

In [76]:
eigvals, eigvecs = K.eigenpairs_left()
eigfuncs = obs(C_test).flatten(1).to(torch.complex128) @ eigvecs.T

torch.save(eigvals, f"generated data/{ds_train_name}_full_edmd_eigvals.pt")
torch.save(eigfuncs, f"generated data/{ds_train_name}_full_edmd_eigfuncs.pt")